In [ ]:
import numpy as np
from gfsupg.solver import CartesianGeometry, FiniteElement1D, Scipy2DFEM
from gfsupg.solver import DeC, ImplicitEuler

from gfsupg.plotting import *

from gfsupg.problem import *

import time
import scipy.sparse as sp

import matplotlib.pyplot as plt

In [ ]:

#problem.T_fin = 1.
order=2

FEM1Dx = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
FEM1Dy = FiniteElement1D(order-1,"gaussLobatto","gaussLobatto")
dec = DeC((order+1)//2,order,"gaussLobatto")
# dec = DeC(4,5,"gaussLobatto")


In [ ]:
# problem = SmoothVortexTestCase(is_long=False, pert_coeff=1e-3, pert_type="opt") #LinearAdvection("smooth_vortex_long")
# problem = StommelGyreTestCase(is_long=False, pert_coeff=1e-3, pert_type="num") #LinearAdvection("smooth_vortex_long")
#problem = SourceVortexTestCase(is_long=True) #LinearAdvection("smooth_vortex_long")
problem = CoriolisVortexTestCase(is_long=True)

In [ ]:

Ns = np.array([4,4], dtype=np.int32)

geom = CartesianGeometry(problem.xL,problem.xR, Ns, problem.geometry_folder, BC=problem.BC)

In [ ]:
FEM2D = Scipy2DFEM(geom,FEM1Dx, FEM1Dy, folder=problem.folderName)

Assembled matrices in 0.051 seconds


In [ ]:
solver = ImplicitEuler(problem, FEM2D, dec, GF = False, stab = "SUPG", trick_second_der=False)

In [20]:
A, B = solver.build_whole_matrices(0.05, 0.01)

In [8]:
A.shape

(75, 75)

In [9]:
B.shape

(75, 75)

In [10]:
q_now  = dict()
q      = np.zeros((len(solver.problem.vars), solver.FEM2D.n_dof_tot)) 
for var in solver.problem.vars:
    q_now[var]  = np.zeros((1, solver.FEM2D.n_dof_tot))
    for i in range(1): #range(self.DeC.n_subNodes):
        q_now[var][i,:] = solver.ic_vect[var]


In [11]:
size_array = sum(np.array([q_now[k].shape[1] for k in solver.problem.vars]))            
vect_q = np.empty((q_now['u'].shape[0], size_array))

solver.build_whole_q_vector(q_now, vect_q)

(1, 25)
(1, 25)
(1, 25)


In [12]:
vect_q.shape

(1, 75)

In [15]:
A @ vect_q[0,:]

array([ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  1.38777878e-15,  4.05762544e-03,  1.67788414e-09,
       -4.05762465e-03,  0.00000000e+00,  4.90026547e-10,  1.81123354e-02,
        5.89308038e-04, -1.78339613e-02,  0.00000000e+00, -1.14024050e-09,
        3.40988818e-03, -1.37125811e-03, -4.05763547e-03,  0.00000000e+00,
       -5.19168042e-14, -2.94885535e-08, -6.24272682e-08, -4.92508812e-13,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  1.36002321e-15, -4.05762437e-03,
       -1.78339664e-02, -4.05762465e-03,  0.00000000e+00,  1.27406904e-09,
        2.59811446e-04, -4.32159228e-04, -1.11581821e-08,  0.00000000e+00,
        4.44693793e-09,  4.96445570e-03,  1.63255820e-02,  4.05758570e-03,
        0.00000000e+00,  5.78426196e-14,  1.17954214e-08, -1.96199986e-08,
       -5.06580888e-13,  0.00000000e+00,  6.25000000e-02,  5.43847491e-02,
        6.24999966e-02,  

In [16]:
indices = [1,2,3]

In [32]:
solver.problem.dirichlet

{'all': ('u', 'v', 'p')}

In [ ]:
from gfsupg.solver import Dirichlet_BC_set

In [ ]:
if solver.problem.dirichlet is not None:
    dirichlet_BC = dict()
    for bc_item in solver.problem.dirichlet.keys():
        BC_values = dict()
        idxs = solver.FEM2D.dirichlet_indexes[bc_item]
        for var in solver.problem.dirichlet[bc_item]:
            BC_values[var] = solver.ic_vect[var][idxs]
        dirichlet_BC[bc_item] = Dirichlet_BC_set(idxs, BC_values)

In [ ]:
A, B = solver.build_whole_matrices(0.01, 0.001, dirichlet_BC)

In [ ]:
A.todense()

matrix([[0.0625, 0.    , 0.    , ..., 0.    , 0.    , 0.    ],
        [0.    , 0.0625, 0.    , ..., 0.    , 0.    , 0.    ],
        [0.    , 0.    , 0.0625, ..., 0.    , 0.    , 0.    ],
        ...,
        [0.    , 0.    , 0.    , ..., 0.0625, 0.    , 0.    ],
        [0.    , 0.    , 0.    , ..., 0.    , 0.0625, 0.    ],
        [0.    , 0.    , 0.    , ..., 0.    , 0.    , 0.0625]],
       shape=(75, 75))

: 